## MNIST Dataset Training

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
fashion_mnist=keras.datasets.fashion_mnist
(x_train_full, y_train_full), (x_test, y_test)=fashion_mnist.load_data()

In [ ]:
print(x_train_full.shape)

In [ ]:
print(x_test.shape)

In [ ]:
print(x_train_full.dtype)

- Each pixel intensity is represented as a byte(0 to 255) which is not good because ideally we would want to `scale it down` to 0 to 1.

- If we have 255 as the data value, then we may have the `exploding gradient problem` where as we go through the iterations, the weight matrix can get very large if we have such a large input.

In [ ]:
x_valid, x_train=x_train_full[:5000]/255., x_train_full[5000:]/255.
y_valid, y_train=y_train_full[:5000], y_train_full[5000:]
x_test=x_test/255.

In [ ]:
print(x_valid.shape)

In [ ]:
plt.imshow(x_train[0], cmap='binary')
plt.axis('off')
plt.show()

In [ ]:
print(y_train)

In [ ]:
class_names=['T-shirt/top', 'Trouser', 'Pullover', 'Dress','Coat',
             'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

In [ ]:
print(class_names[y_train[0]])

In [ ]:
print(x_train.shape)

In [ ]:
print(y_train.shape)

In [ ]:
print(y_valid.shape)

### Visualization of Dataset

In [ ]:
n_rows=4
n_cols=10
plt.figure(figsize=(n_cols*1.2, n_rows*1.2))

for row in range(n_rows):
    for col in range(n_cols):
        idx=n_cols*row+col
        plt.subplot(n_rows, n_cols, idx+1)
        plt.imshow(x_train[idx], cmap='binary', interpolation='nearest')
        plt.axis('off')
        plt.title(class_names[y_train[idx]], fontsize=12)

plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()

### Hyper-Parameter Optimization

- `Parameters`: weights, bias - something that is learnt during machine learning process.

- `Hyper-parameters`: learning rate, number of layers, neurons in each layer, epochs - manually specified.

#### Grid Search

- GridSearch performs exhaustive search over a specified list of parameters.

- We provide the algorithm with the hyperparameters we would like to experiment with and the values we want to try out.

- In the below code, we get 3x3x3x2=54 combinations and we will run each combination 5 times since we have set `cv=5`.

- Total number of runs=54x5=270.

- We used `neg_mean_squared_error` since the `GridSearchCV()` ranks all the algorithms and specifies which one is the best. We are trying to minimize the error.

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

parameters_grid={
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 500, 1000],
    'colsample_bytree': [0.3, 0.7]
}

model=xgb.XGBRegressor(seed=20)

grid=GridSearchCV(
    estimator=model,
    param_grid=parameters_grid,
    scoring='neg_mean_squared_error',
    cv=2,
    verbose=5
)

# Flatten images: (55000, 28, 28) → (55000, 784)
x_train_flat=x_train.reshape(x_train.shape[0], -1)

grid.fit(x_train_flat, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Estimator:", grid.best_estimator_)
print("Best Score (MSE):", -grid.best_score_)

#### Randomized Search

- Grid search works great if the number of combinations are limited.

- In scenarios when the search space is large, RandomizedSearchCV is preferred.

- The algorithm works by evaluating a select few features of random combinations.

- We have the freedom and control over the number of iterations.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

grid={
    'max_depth': [2, 3, 5, 10, 15],
    'learning_rate': [0.05, 0.1, 0.15, 0.20],
    'n_estimators': [100, 500, 900, 1100, 1500],
    'min_child_weight': [1, 2, 3, 4],
    'booster': ['gbtree', 'gblinear']
}

model=xgb.XGBRegressor()

random_cv=RandomizedSearchCV(
    estimator=model,
    param_distributions=grid,
    cv=5,
    n_iter=50,
    scoring='neg_mean_squared_error',
    verbose=5,
    return_train_score=True
)

x_train_flat=x_train.reshape(x_train.shape[0], -1)

random_cv.fit(x_train_flat, y_train)

print("Best Parameters:", random_cv.best_params_)
print("Best Estimator:", random_cv.best_estimator_)
print("Best Score (MSE):", -random_cv.best_score_)

#### Bayesian Optimization

- Bayesian optimization overcomes the drawbacks of random search algorithms by exploring search spaces in a more efficient manner.

- If a region in the search space appears to be promising(i.e; resulted in a small error), this region should be explored more which increases the chances of achieving better performance.

- We should specify the parameters search space.

In [ ]:
from skopt import BayesSearchCV

search_space={
    'max_depth': (4, 20),
    'n_estimators': (100, 500),
    'learning_rate': (0.01, 1.0, 'log-uniform')
}

model=xgb.XGBRegressor()

bayes_search=BayesSearchCV(
    estimator=model,
    search_spaces=search_space,
    n_iter=50,
    scoring='neg_mean_absolute_error'
)

x_train_flat=x_train.reshape(x_train.shape[0], -1)

bayes_search.fit(x_train_flat, y_train)

y_predict=bayes_search.predict(x_test.reshape(x_test.shape[0], -1))

### Training the Neural Network

In [ ]:
model=keras.models.Sequential()

model.add(keras.Input(shape=(28, 28)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(300, activation='relu'))
model.add(keras.layers.Dense(100, activation='relu'))
model.add(keras.layers.Dense(10, activation='softmax'))

In [ ]:
keras.backend.clear_session()
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
model.layers

In [ ]:
model.summary()

- Number of parameters in Dense layer 1: 300x784+300x1 = 235500

- Number of parameters in Dense layer 2: 100x300+100x1 = 30100

- Number of parameters in Output layer: 10x100+10x1 = 1010

In [ ]:
keras.utils.plot_model(model, 'my_fashion_mnist_model.png', show_shapes=True)

In [ ]:
hidden_1=model.layers[1]
print(hidden_1.name)

In [ ]:
weights, biases=hidden_1.get_weights()

In [ ]:
print(weights)

In [ ]:
print(weights.shape)

- Every neuron has 784 weights and there are 300 neurons.

In [ ]:
print(biases)

In [ ]:
print(biases.shape)

### Compiling the Model

In [ ]:
model.compile(
    # Loss function to optimize
    loss='sparse_categorical_crossentropy',
    # Optimizer used
    optimizer='sgd',
    # Metrics to track during training/testing
    metrics=['accuracy']
)

##### 1. `loss='sparse_categorical_crossentropy'`

* This is the **objective function** your model tries to minimize.
* `sparse_categorical_crossentropy` is used when:

  * You are doing **multi-class classification** (e.g., MNIST digits 0–9).
  * Your labels `y_train` are **integers** (like `3`, `7`, `9`) instead of one-hot encoded vectors (`[0,0,0,1,0,0,0,0,0,0]`).
* Equivalent to `categorical_crossentropy`, but **expects integer targets**.
* Formula (per sample):

  $$
  L = -\log \big( p_{model}(y_{true} \mid x) \big)
  $$

  where $p_{model}(y_{true} \mid x)$ is the predicted probability for the correct class.

✅ Example:
If true label = `3` and the model predicts
`[0.01, 0.05, 0.02, 0.80, 0.03, 0.02, 0.02, 0.03, 0.01, 0.01]`,
then loss = `-log(0.80)`.

---

##### 2. `optimizer='sgd'`

* **Optimizer** decides how weights are updated after each batch.

* `'sgd'` = **Stochastic Gradient Descent**.

* It updates weights as:

  $$
  w = w - \eta \cdot \nabla L
  $$

  where:

  * $\eta$ = learning rate
  * $\nabla L$ = gradient of loss wrt weights

* You can also customize it:

  ```python
  keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
  ```

---

##### 3. `metrics=['accuracy']`

* Tells Keras what **extra performance metrics** to compute during training & validation.
* `'accuracy'` means:

  * For classification, it checks if the predicted class (argmax of softmax output) matches the true label.
  * Reported as a percentage.

✅ Example:
If 90 out of 100 predictions are correct → accuracy = 0.90.

---

##### 🔹 Putting it All Together

When you call `compile`, your model is now **ready to train**.

During training:

1. Forward pass → compute predictions.
2. Compute loss (`sparse_categorical_crossentropy`).
3. Backpropagation → compute gradients.
4. Optimizer (`SGD`) updates weights.
5. Accuracy is computed and reported.

In [ ]:
history=model.fit(x_train, y_train, epochs=30, validation_data=(x_valid, y_valid))

In [ ]:
print(history.params)

In [ ]:
print(history.epoch)

In [ ]:
history.history.keys()

In [ ]:
import pandas as pd

pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True)
# Get the current axes
plt.gca().set_ylim(0, 1)
plt.show()

```python
pd.DataFrame(history.history).plot(figsize=(8, 5))
```

* `pd.DataFrame(history.history)` converts the dict to a table where **rows = epochs** and **columns = metrics**.
* `.plot(...)` makes a **line plot** of every column vs. the DataFrame index (epochs).
  With Keras, typical columns are `loss`, `val_loss`, `accuracy`, `val_accuracy`, so you get 2–4 lines, one per metric/variant.
* `figsize=(8, 5)` sets the figure size in inches.

```python
plt.grid(True)
```

Turns on the grid for easier reading.

```python
plt.gca().set_ylim(0, 1)
```

* `plt.gca()` = *get current axes* (the axes created by pandas’ plot).
* `.set_ylim(0, 1)` constrains the **y-axis** to `[0, 1]`.
  This is ideal for **accuracy** (which is in 0–1), but **note**: your **loss** may be >1 early in training, so this line can chop off the loss curve. If you want to see both properly, either:

  * plot only the accuracy columns, or
  * remove/adjust the y-limits, or
  * plot loss on a secondary axis.

```python
plt.show()
```

Renders the plot.

In [ ]:
model.evaluate(x_test, y_test)

- Here the `validation loss` is higher than the training loss, which means there is overfitting and we need to experiment with other optimizers or epochs so that the validation loss will be less than the training loss.

### Testing the Model

In [ ]:
x_new=x_test[:3]

In [ ]:
plt.figure(figsize=(7.2, 2.4))

for idx, image in enumerate(x_new):
    plt.subplot(1, 3, idx+1)
    plt.imshow(image, cmap='binary', interpolation='nearest')

plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()

In [ ]:
y_pred=np.argmax(model.predict(x_new), axis=-1)
print(y_pred)

In [ ]:
print(np.array(class_names)[y_pred])

In [ ]:
plt.figure(figsize=(7.2, 2.4))

for idx, image in enumerate(x_new):
    plt.subplot(1, 3, idx+1)
    plt.imshow(image, cmap='binary', interpolation='nearest')
    plt.axis('off')
    plt.title(class_names[y_test[idx]], fontsize=12)

plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()